In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
import sys, subprocess
# remove the incompatible preinstalled torchao (not needed for this project)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "transformers", "accelerate"])

import os, json, math, random, unicodedata, re, urllib.request
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 38.6 MB/s eta 0:00:00


In [3]:
DATA_DIR  = "/kaggle/working/data"
OUT_DIR   = "/kaggle/working/outputs"
GH_API    = "https://api.github.com/repos/Sidharth-VS/NLP-project/contents/datasets"
PR_CORPUS_CSV    = "corpus_merged.csv"
PR_QUERIES_CSV   = "queries_merged.csv"
PR_RELEVANCE_CSV = "relevance_merged.csv"

MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
MODE       = "lora"          # "lora" | "full"
NORMALIZE_TEXT = True
TASK_INSTRUCT  = "Given a Malayalam Ayurveda question, retrieve the relevant passage"

MAXLEN_Q_TRAIN, MAXLEN_D_TRAIN = 128, 512
EMBED_MAX_LEN = 1024

EPOCHS      = 3
BATCH_SIZE  = 16
LR          = 2e-4 if MODE == "lora" else 1e-5
WARMUP_FRAC = 0.1
TEMPERATURE = 0.05
GRAD_CLIP   = 1.0
GRAD_CKPT   = True
SEED        = 42

HF_USER    = "Nimisha-1"
HF_REPO    = f"{HF_USER}/qwen3-embedding-0.6b-malayalam-ayurveda-{MODE}"
HF_PRIVATE = False

LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# corpus + query columns are set from your files; relevance columns are still auto-detected
COL = dict(corpus_id="passage_id", corpus_text="passage_text_ml", corpus_title=None,
           query_id="query_id", query_text="query_text_ml",
           rel_qid=None, rel_did=None, rel_score=None, rel_split=None)

os.makedirs(DATA_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_BF16 = DEV.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("device:", DEV, "| amp dtype:", AMP_DTYPE)

device: cuda | amp dtype: torch.float16


In [4]:
def _gh_list(url, depth=0):
    req = urllib.request.Request(url, headers={"User-Agent": "kaggle-nb"})
    with urllib.request.urlopen(req) as r:
        items = json.load(r)
    out = []
    for it in items:
        if it["type"] == "file":
            out.append(it)
        elif it["type"] == "dir" and depth < 3:
            out += _gh_list(it["url"], depth + 1)
    return out

_GH_CACHE = None
def ensure_file(name):
    global _GH_CACHE
    path = os.path.join(DATA_DIR, name)
    if os.path.exists(path):
        return path
    if _GH_CACHE is None:
        _GH_CACHE = _gh_list(GH_API)
    hit = [i for i in _GH_CACHE if i["name"] == name]
    if not hit:
        raise FileNotFoundError(f"{name} not found locally or on GitHub")
    urllib.request.urlretrieve(hit[0]["download_url"], path)
    print("downloaded", name)
    return path

def norm_text(s):
    s = "" if pd.isna(s) else str(s)
    if NORMALIZE_TEXT:
        s = unicodedata.normalize("NFC", s)
        s = s.replace("\u200b", "").replace("\ufeff", "")
        s = re.sub(r"\s+", " ", s).strip()
    return s

def pick(df, override, cands, required=True, what=""):
    if override:
        return override
    low = {c.lower(): c for c in df.columns}
    for c in cands:
        if c.lower() in low:
            return low[c]
    if required:
        raise KeyError(f"Could not detect column for {what}. Columns: {list(df.columns)}. Set COL[...] manually.")
    return None

corpus  = pd.read_csv(ensure_file(PR_CORPUS_CSV))
queries = pd.read_csv(ensure_file(PR_QUERIES_CSV))
rel     = pd.read_csv(ensure_file(PR_RELEVANCE_CSV))
print("corpus cols :", list(corpus.columns), len(corpus))
print("queries cols:", list(queries.columns), len(queries))
print("rel cols    :", list(rel.columns), len(rel))

c_id  = pick(corpus, COL["corpus_id"], ["_id", "id", "doc_id", "corpus_id", "corpus-id", "passage_id"], what="corpus id")
c_txt = pick(corpus, COL["corpus_text"], ["text", "passage", "content", "document"], what="corpus text")
c_ttl = pick(corpus, COL["corpus_title"], ["title"], required=False)
q_id  = pick(queries, COL["query_id"], ["_id", "id", "query_id", "query-id", "qid"], what="query id")
q_txt = pick(queries, COL["query_text"], ["text", "query", "question"], what="query text")
r_q   = pick(rel, COL["rel_qid"], ["query-id", "query_id", "qid", "query"], what="relevance query id")
r_d   = pick(rel, COL["rel_did"], ["corpus-id", "corpus_id", "doc_id", "docid", "passage_id", "doc"], what="relevance doc id")
r_s   = pick(rel, COL["rel_score"], ["score", "relevance", "label", "rel"], required=False)
r_sp  = pick(rel, COL["rel_split"], ["split", "set", "subset"], required=False)

ctext = {}
for _, row in corpus.iterrows():
    t = norm_text(row[c_txt])
    if c_ttl and not pd.isna(row[c_ttl]):
        t = norm_text(row[c_ttl]) + ". " + t
    ctext[str(row[c_id])] = t
qtext = {str(r[q_id]): norm_text(r[q_txt]) for _, r in queries.iterrows()}

rel = rel.copy()
rel[r_q] = rel[r_q].astype(str); rel[r_d] = rel[r_d].astype(str)
if r_s:
    rel = rel[pd.to_numeric(rel[r_s], errors="coerce").fillna(0) > 0]
rel = rel[rel[r_q].isin(qtext) & rel[r_d].isin(ctext)]

def _norm_split(s):
    s = str(s).lower().strip()
    return {"dev": "validation", "val": "validation", "valid": "validation", "eval": "validation"}.get(s, s)

if r_sp:
    rel["_split"] = rel[r_sp].map(_norm_split)
elif "split" in [c.lower() for c in queries.columns]:
    sc = [c for c in queries.columns if c.lower() == "split"][0]
    m = {str(a): _norm_split(b) for a, b in zip(queries[q_id], queries[sc])}
    rel["_split"] = rel[r_q].map(m)
else:
    print("!! No split column found -> random 80/10/10 split by query (seed fixed)")
    qids = sorted(rel[r_q].unique()); random.Random(SEED).shuffle(qids)
    n = len(qids); a, b = int(.8 * n), int(.9 * n)
    m = {q: ("train" if i < a else "validation" if i < b else "test") for i, q in enumerate(qids)}
    rel["_split"] = rel[r_q].map(m)

QRELS = {"train": {}, "validation": {}, "test": {}}
for q, d, s in zip(rel[r_q], rel[r_d], rel["_split"]):
    if s in QRELS:
        QRELS[s].setdefault(q, set()).add(d)
for s in QRELS:
    print(f"{s:10s}: {len(QRELS[s])} queries, {sum(len(v) for v in QRELS[s].values())} pairs")
assert QRELS["train"] and QRELS["test"], "Need non-empty train and test splits"

train_pairs = [(q, d) for q, ds in QRELS["train"].items() for d in ds]
CORPUS_IDS = list(ctext.keys())

downloaded corpus_merged.csv
downloaded queries_merged.csv
downloaded relevance_merged.csv
corpus cols : ['passage_id', 'source_id', 'source_title', 'passage_text_ml', 'chapter_title', 'printed_page_start', 'printed_page_end', 'pdf_page_start', 'pdf_page_end', 'passage_topic_ml', 'passage_content_type', 'source_text_status', 'text_normalization', 'related_query_ids', 'corpus_split', 'review_notes'] 398
queries cols: ['query_id', 'task_type', 'language', 'domain', 'query_text_ml', 'query_type', 'query_topic_ml', 'expected_information_ml', 'gold_passage_ids', 'gold_relevance_levels', 'difficulty', 'source_title', 'source_chapter', 'source_printed_pages', 'source_pdf_pages', 'source_evidence_status', 'query_generation_method', 'validation_status', 'reviewer_notes', 'split'] 420
rel cols    : ['query_id', 'passage_id', 'relevance_grade', 'relevance_rationale_ml', 'evidence_type', 'evidence_status', 'validation_status', 'reviewer_notes'] 420
train     : 140 queries, 140 pairs
validation: 30

In [5]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
EOS_ID = tok.convert_tokens_to_ids("<|endoftext|>")
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else EOS_ID

def fmt_query(q):
    return f"Instruct: {TASK_INSTRUCT}\nQuery: {q}"

def make_batch(texts, max_len):
    ids = tok(texts, add_special_tokens=False, truncation=True, max_length=max_len - 1)["input_ids"]
    ids = [x + [EOS_ID] for x in ids]
    L = max(len(x) for x in ids)
    inp = torch.full((len(ids), L), PAD_ID, dtype=torch.long)
    att = torch.zeros((len(ids), L), dtype=torch.long)
    for i, x in enumerate(ids):                       # left padding -> last token = EOS
        inp[i, L - len(x):] = torch.tensor(x)
        att[i, L - len(x):] = 1
    return inp.to(DEV), att.to(DEV)

def embed(model, texts, max_len):
    inp, att = make_batch(texts, max_len)
    with torch.autocast(device_type=DEV.type, dtype=AMP_DTYPE, enabled=DEV.type == "cuda"):
        h = model(input_ids=inp, attention_mask=att, use_cache=False).last_hidden_state
    return F.normalize(h[:, -1].float(), dim=-1)

@torch.no_grad()
def encode(model, texts, max_len, bs=16, desc="encode"):
    model.eval()
    order = np.argsort([-len(t) for t in texts])
    out = torch.zeros(len(texts), model.config.hidden_size)
    for i in tqdm(range(0, len(texts), bs), desc=desc, leave=False):
        idx = order[i:i + bs]
        out[torch.as_tensor(idx)] = embed(model, [texts[j] for j in idx], max_len).cpu()
    return out

def retrieval_metrics(q_emb, d_emb, qids, qrels, ks=(1, 5, 10)):
    q_emb, d_emb = q_emb.to(DEV), d_emb.to(DEV)
    sims = q_emb @ d_emb.T
    top = sims.topk(max(ks), dim=1).indices.cpu().numpy()
    res = {f"Recall@{k}": [] for k in ks}; res["MRR@10"] = []; res["nDCG@10"] = []
    for i, q in enumerate(qids):
        gold = qrels[q]
        ranked = [CORPUS_IDS[j] for j in top[i]]
        for k in ks:
            res[f"Recall@{k}"].append(len(gold & set(ranked[:k])) / len(gold))
        rr = next((1 / (r + 1) for r, d in enumerate(ranked[:10]) if d in gold), 0.0)
        res["MRR@10"].append(rr)
        dcg = sum(1 / math.log2(r + 2) for r, d in enumerate(ranked[:10]) if d in gold)
        idcg = sum(1 / math.log2(r + 2) for r in range(min(len(gold), 10)))
        res["nDCG@10"].append(dcg / idcg)
    return {k: float(np.mean(v)) for k, v in res.items()}

def evaluate(model, split, save_prefix=None):
    qids = list(QRELS[split].keys())
    q_emb = encode(model, [fmt_query(qtext[q]) for q in qids], EMBED_MAX_LEN, desc=f"{split} queries")
    d_emb = encode(model, [ctext[d] for d in CORPUS_IDS], EMBED_MAX_LEN, bs=8, desc="corpus")
    if save_prefix:
        np.save(f"{OUT_DIR}/{save_prefix}_{split}_query_emb.npy", q_emb.numpy())
        np.save(f"{OUT_DIR}/{save_prefix}_corpus_emb.npy", d_emb.numpy())
        json.dump({"query_ids": qids, "corpus_ids": CORPUS_IDS}, open(f"{OUT_DIR}/{save_prefix}_ids.json", "w"))
    return retrieval_metrics(q_emb, d_emb, qids, QRELS[split])

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.71k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

In [6]:
model = AutoModel.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEV)
print("params (M):", sum(p.numel() for p in model.parameters()) / 1e6)

RESULTS = {}
RESULTS["base_zero_shot"] = evaluate(model, "test", save_prefix="base")
print("BASE (zero-shot) TEST:", json.dumps(RESULTS["base_zero_shot"], indent=2))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

params (M): 595.776512


test queries:   0%|          | 0/2 [00:00<?, ?it/s]

corpus:   0%|          | 0/50 [00:00<?, ?it/s]

BASE (zero-shot) TEST: {
  "Recall@1": 0.4666666666666667,
  "Recall@5": 0.6333333333333333,
  "Recall@10": 0.6333333333333333,
  "MRR@10": 0.5305555555555556,
  "nDCG@10": 0.5564178688405437
}


In [7]:
if GRAD_CKPT:
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if MODE == "lora":
        model.enable_input_require_grads()

if MODE == "lora":
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                     target_modules=LORA_TARGETS, bias="none", task_type="FEATURE_EXTRACTION")
    model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
else:
    for p in model.parameters():
        p.requires_grad = True

trainable params: 10,092,544 || all params: 605,869,056 || trainable%: 1.6658


In [8]:
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
steps_per_epoch = math.ceil(len(train_pairs) / BATCH_SIZE)
sched = get_cosine_schedule_with_warmup(opt, int(WARMUP_FRAC * steps_per_epoch * EPOCHS), steps_per_epoch * EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=(DEV.type == "cuda" and not USE_BF16))

def snapshot():
    return {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}

best_mrr, best_state, history = -1.0, None, []
has_val = len(QRELS["validation"]) > 0

for epoch in range(EPOCHS):
    model.train()
    random.shuffle(train_pairs)
    running = []
    pbar = tqdm(range(0, len(train_pairs), BATCH_SIZE), desc=f"epoch {epoch+1}/{EPOCHS}")
    for i in pbar:
        batch = train_pairs[i:i + BATCH_SIZE]
        if len(batch) < 2:
            continue
        qs   = [fmt_query(qtext[q]) for q, _ in batch]
        ds   = [ctext[d] for _, d in batch]
        dids = [d for _, d in batch]
        qe = embed(model, qs, MAXLEN_Q_TRAIN)
        de = embed(model, ds, MAXLEN_D_TRAIN)
        logits = qe @ de.T / TEMPERATURE
        # mask in-batch docs that are actually positives for another query
        mask = torch.zeros_like(logits, dtype=torch.bool)
        for a, (q, _) in enumerate(batch):
            for b in range(len(batch)):
                if a != b and dids[b] in QRELS["train"][q]:
                    mask[a, b] = True
        logits = logits.masked_fill(mask, -1e4)
        labels = torch.arange(len(batch), device=DEV)
        loss = F.cross_entropy(logits, labels)

        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        scaler.step(opt); scaler.update(); sched.step()
        running.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(running[-20:]):.4f}")

    rec = {"epoch": epoch + 1, "train_loss": float(np.mean(running))}
    if has_val:
        vm = evaluate(model, "validation")
        rec.update({f"val_{k}": v for k, v in vm.items()})
        if vm["MRR@10"] > best_mrr:
            best_mrr, best_state = vm["MRR@10"], snapshot()
            rec["best"] = True
    history.append(rec); print(rec)

if best_state is not None:
    for n, p in model.named_parameters():
        if n in best_state:
            p.data.copy_(best_state[n].to(p.device))
    print(f"restored best epoch (val MRR@10={best_mrr:.4f})")

/tmp/ipykernel_23/1208567002.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(DEV.type == "cuda" and not USE_BF16))


epoch 1/3:   0%|          | 0/9 [00:00<?, ?it/s]

validation queries:   0%|          | 0/2 [00:00<?, ?it/s]

corpus:   0%|          | 0/50 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 0.67685859484805, 'val_Recall@1': 0.6333333333333333, 'val_Recall@5': 0.8666666666666667, 'val_Recall@10': 0.9333333333333333, 'val_MRR@10': 0.7533730158730159, 'val_nDCG@10': 0.7978125577182602, 'best': True}


...

{'epoch': 2, 'train_loss': 0.27235151247845757, 'val_Recall@1': 0.8, 'val_Recall@5': 0.9333333333333333, 'val_Recall@10': 0.9666666666666667, 'val_MRR@10': 0.8544444444444445, 'val_nDCG@10': 0.8812592380231782, 'best': True}


epoch 3/3:   0%|          | 0/9 [00:00<?, ?it/s]

validation queries:   0%|          | 0/2 [00:00<?, ?it/s]

corpus:   0%|          | 0/50 [00:00<?, ?it/s]

{'epoch': 3, 'train_loss': 0.05971761353106962, 'val_Recall@1': 0.8, 'val_Recall@5': 0.9, 'val_Recall@10': 0.9333333333333333, 'val_MRR@10': 0.8492063492063492, 'val_nDCG@10': 0.8698397613492083}
restored best epoch (val MRR@10=0.8544)


In [9]:
RESULTS[f"finetuned_{MODE}"] = evaluate(model, "test", save_prefix=f"ft_{MODE}")
model.save_pretrained(f"{OUT_DIR}/qwen3_emb_{MODE}")
tok.save_pretrained(f"{OUT_DIR}/qwen3_emb_{MODE}")

df = pd.DataFrame(RESULTS).T.round(4)
df.loc["delta (ft - base)"] = (df.loc[f"finetuned_{MODE}"] - df.loc["base_zero_shot"]).round(4)
print(df)
df.to_csv(f"{OUT_DIR}/test_results_{MODE}.csv")
json.dump({"results": RESULTS, "history": history,
           "config": dict(model=MODEL_NAME, mode=MODE, lr=LR, epochs=EPOCHS, batch=BATCH_SIZE,
                          temp=TEMPERATURE, lora_r=LORA_R)},
          open(f"{OUT_DIR}/run_{MODE}.json", "w"), indent=2, ensure_ascii=False)
print("saved to", OUT_DIR)

test queries:   0%|          | 0/2 [00:00<?, ?it/s]

corpus:   0%|          | 0/50 [00:00<?, ?it/s]

                   Recall@1  Recall@5  Recall@10  MRR@10  nDCG@10
base_zero_shot       0.4667    0.6333     0.6333  0.5306   0.5564
finetuned_lora       0.8000    0.8000     0.8000  0.8000   0.8000
delta (ft - base)    0.3333    0.1667     0.1667  0.2694   0.2436
saved to /kaggle/working/outputs


In [10]:
import shutil
from huggingface_hub import HfApi, login

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("finetune_mal")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "HF_TOKEN not found. Add it as a Kaggle Secret named HF_TOKEN (write access)."
login(token=HF_TOKEN)

PUSH_DIR = f"{OUT_DIR}/hf_push_{MODE}"
shutil.rmtree(PUSH_DIR, ignore_errors=True); os.makedirs(PUSH_DIR)

if MODE == "lora":
    shutil.copytree(f"{OUT_DIR}/qwen3_emb_lora", f"{PUSH_DIR}/lora_adapter")   # small adapter
    merged = model.merge_and_unload()                                          # standalone model
    merged.save_pretrained(PUSH_DIR, safe_serialization=True)
else:
    model.save_pretrained(PUSH_DIR, safe_serialization=True)
tok.save_pretrained(PUSH_DIR)

try:
    tbl = df.to_markdown()
except Exception:
    tbl = "```\n" + df.to_string() + "\n```"

card = f"""---
base_model: {MODEL_NAME}
language: [ml]
library_name: transformers
tags: [sentence-similarity, feature-extraction, embeddings, malayalam, ayurveda, {MODE}]
---
# {HF_REPO.split('/')[1]}

`{MODEL_NAME}` fine-tuned ({MODE}) with an InfoNCE contrastive loss on a Malayalam Ayurveda
passage-retrieval dataset (https://github.com/Sidharth-VS/NLP-project).

- Pooling: last token (left padding), L2-normalised embeddings
- Query format: `Instruct: {TASK_INSTRUCT}\\nQuery: <query>`; documents are encoded without an instruction
- Training: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LR}, temperature={TEMPERATURE}, LoRA r={LORA_R if MODE=='lora' else 'n/a'}

## Test-set results (zero-shot base vs fine-tuned)
{tbl}

## Usage
```python
import torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
tok = AutoTokenizer.from_pretrained("{HF_REPO}", padding_side="left")
model = AutoModel.from_pretrained("{HF_REPO}").eval()
def emb(texts):
    b = tok(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    with torch.no_grad():
        h = model(**b).last_hidden_state[:, -1]
    return F.normalize(h, dim=-1)
```
"""
open(f"{PUSH_DIR}/README.md", "w", encoding="utf-8").write(card)
shutil.copy(f"{OUT_DIR}/test_results_{MODE}.csv", f"{PUSH_DIR}/test_results_{MODE}.csv")

api = HfApi()
api.create_repo(HF_REPO, repo_type="model", private=HF_PRIVATE, exist_ok=True)
api.upload_folder(folder_path=PUSH_DIR, repo_id=HF_REPO, repo_type="model",
                  commit_message=f"Qwen3-Embedding {MODE} fine-tuned on Malayalam Ayurveda retrieval")
print(f"Uploaded: https://huggingface.co/{HF_REPO}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Uploaded: https://huggingface.co/Nimisha-1/qwen3-embedding-0.6b-malayalam-ayurveda-lora
